In [8]:
# Setup
import os
os.chdir('/Users/anshtomar/PROJECTS/Credit-Risk-Scorecard')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv('data/cs_clean.csv')
print(df.shape)
print("Data loaded!")


(149999, 12)
Data loaded!


In [9]:
# Step 2 - Define target and features
target = 'SeriousDlqin2yrs'

features = [col for col in df.columns 
            if col != target 
            and col != 'age_group']

print("Target:", target)
print("Features:", features)
print("Total features:", len(features))


Target: SeriousDlqin2yrs
Features: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']
Total features: 10


In [10]:
# Step 2 - Define target and features
target = 'SeriousDlqin2yrs'

features = [col for col in df.columns 
            if col != target 
            and col != 'age_group']

print("Target:", target)
print("Features:", features)
print("Total features:", len(features))


Target: SeriousDlqin2yrs
Features: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents']
Total features: 10


In [11]:
# Step 3 - Bin a feature into groups
def bin_feature(df, feature, n_bins=10):
    df[feature + '_bin'] = pd.qcut(
        df[feature], 
        q=n_bins, 
        duplicates='drop'
    )
    return df

# Test on age first
df = bin_feature(df, 'age')
print(df['age_bin'].value_counts().sort_index())


age_bin
(20.999, 33.0]    17084
(33.0, 39.0]      14919
(39.0, 44.0]      15799
(44.0, 48.0]      14741
(48.0, 52.0]      14826
(52.0, 56.0]      14214
(56.0, 61.0]      16878
(61.0, 65.0]      12939
(65.0, 72.0]      14258
(72.0, 109.0]     14341
Name: count, dtype: int64


## 📝 What is Binning? (Step 3)

Instead of using age as a raw number (45, 32, 67),
we group it into buckets:
- Bucket 1: age 20-33
- Bucket 2: age 33-39
- Bucket 3: age 39-44
...and so on

Then we calculate how many defaulters are 
in each bucket — that's WoE!

Why do this?
- Raw numbers are hard to interpret
- Buckets make patterns clearer
- Handles outliers automatically
- This is how real banks process data


In [12]:
# Step 4 - Calculate WoE for age
def calculate_woe(df, feature_bin, target):
    temp = df.groupby(feature_bin)[target].agg(
        ['sum', 'count']
    )
    temp.columns = ['defaults', 'total']
    temp['non_defaults'] = temp['total'] - temp['defaults']
    
    total_defaults = temp['defaults'].sum()
    total_non_defaults = temp['non_defaults'].sum()
    
    temp['woe'] = np.log(
        (temp['defaults'] / total_defaults) /
        (temp['non_defaults'] / total_non_defaults)
    )
    
    return temp

age_woe = calculate_woe(df, 'age_bin', target)
print(age_woe[['defaults', 'total', 'woe']])


                defaults  total       woe
age_bin                                  
(20.999, 33.0]      1940  17084  0.581351
(33.0, 39.0]        1430  14919  0.392068
(39.0, 44.0]        1365  15799  0.277836
(44.0, 48.0]        1200  14741  0.212867
(48.0, 52.0]        1142  14826  0.152822
(52.0, 56.0]         935  14214 -0.017125
(56.0, 61.0]         839  16878 -0.314300
(61.0, 65.0]         485  12939 -0.609380
(65.0, 72.0]         380  14258 -0.961621
(72.0, 109.0]        310  14341 -1.176184


## 📝 WoE Results for Age

Positive WoE = High Risk (more defaults)
Negative WoE = Low Risk (fewer defaults)

Key finding:
- Age 20-33: WoE = +0.58 (riskiest group)
- Age 72+: WoE = -1.17 (safest group)

This confirms age is a strong predictor!
Younger borrowers are significantly 
riskier than older borrowers.


In [13]:
# Step 5 - Calculate IV for Age

# Count defaults per bin
temp = df.groupby('age_bin')['SeriousDlqin2yrs'].agg(['sum','count'])
temp.columns = ['defaults', 'total']
temp['non_defaults'] = temp['total'] - temp['defaults']

# Calculate percentages
temp['pct_defaults'] = temp['defaults'] / temp['defaults'].sum()
temp['pct_non_defaults'] = temp['non_defaults'] / temp['non_defaults'].sum()

# Calculate WoE
temp['woe'] = np.log(temp['pct_defaults'] / temp['pct_non_defaults'])

# Calculate IV
temp['iv'] = (temp['pct_defaults'] - temp['pct_non_defaults']) * temp['woe']

# Total IV
iv_age = temp['iv'].sum()
print(f"IV for Age = {iv_age:.4f}")

if iv_age >= 0.3:
    print("Strong predictor ⭐ Keep it!")
elif iv_age >= 0.1:
    print("Medium predictor — Keep it")
elif iv_age >= 0.02:
    print("Weak predictor")
else:
    print("Useless — Drop it!")


IV for Age = 0.2592
Medium predictor — Keep it


## 📝 IV Result for Age

IV = 0.2592 → Medium Predictor → KEEP ✅

IV Interpretation:
- Less than 0.02 → Useless, drop it
- 0.02 to 0.1  → Weak
- 0.1 to 0.3   → Medium, keep it
- Above 0.3    → Strong ⭐

Age IV = 0.2592 confirms age is a 
useful predictor of credit default.


In [14]:
# Step 6 - Calculate IV for all features
iv_scores = {}

for feature in features:
    # Bin the feature
    df[feature + '_bin'] = pd.qcut(
        df[feature], q=10, duplicates='drop'
    )
    
    # Calculate IV
    temp = df.groupby(feature + '_bin')['SeriousDlqin2yrs'].agg(['sum','count'])
    temp.columns = ['defaults', 'total']
    temp['non_defaults'] = temp['total'] - temp['defaults']
    temp['pct_defaults'] = temp['defaults'] / temp['defaults'].sum()
    temp['pct_non_defaults'] = temp['non_defaults'] / temp['non_defaults'].sum()
    temp['woe'] = np.log(temp['pct_defaults'] / temp['pct_non_defaults'])
    temp['iv'] = (temp['pct_defaults'] - temp['pct_non_defaults']) * temp['woe']
    
    iv_scores[feature] = round(temp['iv'].sum(), 4)

# Print results
iv_df = pd.Series(iv_scores).sort_values(ascending=False)
print("IV SCORES FOR ALL FEATURES")
print("="*45)
print(iv_df)


IV SCORES FOR ALL FEATURES
RevolvingUtilizationOfUnsecuredLines    1.1134
NumberOfTime30-59DaysPastDueNotWorse    0.4718
age                                     0.2592
DebtRatio                               0.0737
MonthlyIncome                           0.0670
NumberOfOpenCreditLinesAndLoans         0.0669
NumberOfDependents                      0.0250
NumberRealEstateLoansOrLines            0.0121
NumberOfTimes90DaysLate                 0.0000
NumberOfTime60-89DaysPastDueNotWorse    0.0000
dtype: float64


## 📝 IV Scores — Feature Selection

Top predictors:
1. RevolvingUtilization → IV=1.11 (STRONGEST!)
2. 30-59 DaysLate → IV=0.47 (Strong)
3. Age → IV=0.26 (Medium)

Dropped features (IV < 0.02):
- NumberRealEstateLoans → IV=0.012
- NumberOfTimes90DaysLate → IV=0.000
- NumberOfTime60-89DaysLate → IV=0.000

Why dropped?
These features carry almost no information
about who will default.


In [15]:
# Step 7 - Keep only features with IV >= 0.02
good_features = [f for f in iv_scores 
                 if iv_scores[f] >= 0.02]

print("Features KEPT:")
for f in good_features:
    print(f, "→ IV =", iv_scores[f])

print()
print("Features DROPPED:")
for f in iv_scores:
    if iv_scores[f] < 0.02:
        print(f, "→ IV =", iv_scores[f])


Features KEPT:
RevolvingUtilizationOfUnsecuredLines → IV = 1.1134
age → IV = 0.2592
NumberOfTime30-59DaysPastDueNotWorse → IV = 0.4718
DebtRatio → IV = 0.0737
MonthlyIncome → IV = 0.067
NumberOfOpenCreditLinesAndLoans → IV = 0.0669
NumberOfDependents → IV = 0.025

Features DROPPED:
NumberOfTimes90DaysLate → IV = 0.0
NumberRealEstateLoansOrLines → IV = 0.0121
NumberOfTime60-89DaysPastDueNotWorse → IV = 0.0


## 📝 Feature Selection Result

7 features kept, 3 dropped.

Dropped because IV < 0.02:
- They carry almost zero information
- Keeping them would add noise to model
- This is standard practice in credit scoring

Interview answer:
"I used IV to automatically select features.
Any feature with IV below 0.02 was dropped
as it adds no predictive value."


In [16]:
# Step 8 - Replace actual values with WoE values

df_woe = df[good_features + ['SeriousDlqin2yrs']].copy()

for feature in good_features:
    # Calculate WoE for each bin
    temp = df.groupby(feature + '_bin')['SeriousDlqin2yrs'].agg(['sum','count'])
    temp.columns = ['defaults', 'total']
    temp['non_defaults'] = temp['total'] - temp['defaults']
    temp['pct_defaults'] = temp['defaults'] / temp['defaults'].sum()
    temp['pct_non_defaults'] = temp['non_defaults'] / temp['non_defaults'].sum()
    temp['woe'] = np.log(temp['pct_defaults'] / temp['pct_non_defaults'])
    
    # Create WoE mapping
    woe_map = temp['woe'].to_dict()
    
    # Replace values with WoE
    df_woe[feature] = df[feature + '_bin'].map(woe_map)

print("WoE Transformation done!")
print(df_woe.head())


WoE Transformation done!
  RevolvingUtilizationOfUnsecuredLines       age  \
0                             1.020577  0.212867   
1                             1.020577  0.277836   
2                             0.297965  0.392068   
3                            -0.688368  0.581351   
4                             1.020577  0.152822   

  NumberOfTime30-59DaysPastDueNotWorse DebtRatio MonthlyIncome  \
0                             1.901119  0.578974     -0.359045   
1                            -0.257826  0.021594      0.412732   
2                            -0.257826  0.021594      0.412732   
3                            -0.257826  0.021594      0.412732   
4                            -0.257826 -0.230913     -0.422327   

  NumberOfOpenCreditLinesAndLoans NumberOfDependents  SeriousDlqin2yrs  
0                       -0.041551           0.209356                 1  
1                       -0.046430          -0.088273                 0  
2                        0.514819          -0.

## 📝 WoE Transformation — Step 8

What happened?
Every raw value is replaced with its WoE score.

Example for Age:
- Before: 45, 32, 67 (raw numbers)
- After:  0.21, 0.58, -0.61 (risk scores)

Why do this?
- Logistic Regression works better with 
  WoE values than raw numbers
- All features are now on same scale
- Negative WoE = low risk customer
- Positive WoE = high risk customer

This is exactly how Amex, HDFC and other 
banks prepare data for credit scorecards!


In [17]:
# Step 9 - Save WoE transformed data
df_woe.to_csv('data/cs_woe.csv', index=False)
print("WoE data saved!")
print(f"Shape: {df_woe.shape}")


WoE data saved!
Shape: (149999, 8)
